In [3]:
import numpy as np
import pandas as pd

import statsmodels.api as sm
import statsmodels.formula.api as smf


In [4]:
raw = pd.read_csv("/Users/ronnygottheway/Desktop/美赛/data/raw/2026_MCM_Problem_C_Data.csv")
final = pd.read_csv("/Users/ronnygottheway/Desktop/美赛/data/cleaned_data/final_outputs.csv")

print(raw.shape, final.shape)
print(raw.columns[:10])
print(final.columns)


(421, 53) (2777, 13)
Index(['celebrity_name', 'ballroom_partner', 'celebrity_industry',
       'celebrity_homestate', 'celebrity_homecountry/region',
       'celebrity_age_during_season', 'season', 'results', 'placement',
       'week1_judge1_score'],
      dtype='object')
Index(['celebrity_name', 'season', 'week', 'mean_score', 'popularity_oof',
       'J_total', 'J_mean_person', 'week_weight', 'popularity_week_weighted',
       'placement', 'max_week_person', 'max_week_season', 'if_elim'],
      dtype='object')


In [5]:
keep_cols = [
    "celebrity_name", "season",
    "ballroom_partner",
    "celebrity_industry",
    "celebrity_homestate",
    "celebrity_homecountry/region",
    "celebrity_age_during_season",
]
feat = raw[keep_cols].copy()

df = final.merge(feat, on=["celebrity_name", "season"], how="left")

# 基础检查：不应出现大量缺失的 ballroom_partner/industry
print(df[["ballroom_partner", "celebrity_industry", "celebrity_age_during_season"]].isna().mean())


ballroom_partner               0.002521
celebrity_industry             0.002521
celebrity_age_during_season    0.002521
dtype: float64


In [6]:
df = df.sort_values(["season", "celebrity_name", "week"]).copy()

# 每个选手-赛季的淘汰周（最早 if_elim==1 的 week；若从未淘汰 -> inf）
elim_week = (
    df.loc[df["if_elim"] == 1]
      .groupby(["season", "celebrity_name"])["week"]
      .min()
      .rename("elim_week")
)

df = df.merge(elim_week, on=["season", "celebrity_name"], how="left")
df["elim_week"] = df["elim_week"].fillna(np.inf)

df["alive"] = df["week"] <= df["elim_week"]

# 删除淘汰后周（避免 0 分污染）
df_model = df.loc[df["alive"]].copy()

print("Before:", df.shape, "After (drop post-elim):", df_model.shape)


Before: (2777, 20) After (drop post-elim): (2777, 20)


In [7]:
eps = 1e-12

df_model["fan_value"] = df_model["popularity_week_weighted"].astype(float)

# 平滑：防止出现全0导致分母为0
df_model["fan_value"] = df_model["fan_value"].clip(lower=eps)

# 同一 (season, week) 内归一化
den = df_model.groupby(["season", "week"])["fan_value"].transform("sum")
df_model["fan_share_t"] = df_model["fan_value"] / den

# 检查：每个 (season, week) 的份额和应≈1
check = df_model.groupby(["season", "week"])["fan_share_t"].sum().reset_index()
print(check["fan_share_t"].describe())


count    3.350000e+02
mean     1.000000e+00
std      4.505244e-17
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
Name: fan_share_t, dtype: float64


In [8]:
p = df_model["fan_share_t"].clip(eps, 1 - eps)
df_model["fan_logit"] = np.log(p / (1 - p))


In [11]:
df_model["celeb_season_id"] = df_model["celebrity_name"].astype(str) + "_S" + df_model["season"].astype(str)

# 简单地区合并：优先用州，否则用国家/地区
df_model["homeland"] = df_model["celebrity_homestate"].fillna(df_model["celebrity_homecountry/region"])

# 强制类别类型（便于 C() 处理）
for c in ["celebrity_industry", "homeland", "ballroom_partner", "season", "week"]:
    df_model[c] = df_model[c].astype("category")


In [16]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ====== 合并稀有类别：homeland、industry（阈值可调）======
def collapse_rare(series: pd.Series, min_count=10, other_name="Other"):
    vc = series.value_counts(dropna=False)
    keep = vc[vc >= min_count].index
    return series.where(series.isin(keep), other_name)

df_model2 = df_model2.copy()

df_model2["homeland2"] = collapse_rare(df_model2["homeland"].astype(str), min_count=10)
df_model2["industry2"] = collapse_rare(df_model2["celebrity_industry"].astype(str), min_count=10)

# 统一 category 类型
for c in ["season", "week", "homeland2", "industry2", "ballroom_partner"]:
    df_model2[c] = df_model2[c].astype("category")

print("levels:",
      "industry2", df_model2["industry2"].nunique(),
      "homeland2", df_model2["homeland2"].nunique(),
      "partner", df_model2["ballroom_partner"].nunique(),
      "season", df_model2["season"].nunique(),
      "week", df_model2["week"].nunique())



levels: industry2 21 homeland2 51 partner 60 season 34 week 11


In [18]:
def fit_judge_models(data):
    """
    依次尝试：
    1) MixedLM + vc(partner) + week
    2) MixedLM + vc(partner)（去掉 week）
    3) MixedLM（不加 vc）+ week
    4) MixedLM（不加 vc）（去掉 week）
    5) OLS + partner 固定效应 + cluster(season_id)（最终兜底，最稳）
    """
    results = {}

    # —— 1) MixedLM + vc + week
    f1 = "J_total ~ celebrity_age_during_season + C(industry2) + C(homeland2) + C(season) + C(week)"
    try:
        md = smf.mixedlm(
            f1, data=data,
            groups=data["celeb_season_id"].values,
            vc_formula={"partner_re": "0 + C(ballroom_partner)"}
        )
        res = md.fit(reml=True, method="lbfgs")
        results["judge_best"] = ("MixedLM_vc_week", f1, res)
        return results
    except Exception as e:
        results["judge_try1_error"] = repr(e)

    # —— 2) MixedLM + vc（去 week）
    f2 = "J_total ~ celebrity_age_during_season + C(industry2) + C(homeland2) + C(season)"
    try:
        md = smf.mixedlm(
            f2, data=data,
            groups=data["celeb_season_id"].values,
            vc_formula={"partner_re": "0 + C(ballroom_partner)"}
        )
        res = md.fit(reml=True, method="lbfgs")
        results["judge_best"] = ("MixedLM_vc_no_week", f2, res)
        return results
    except Exception as e:
        results["judge_try2_error"] = repr(e)

    # —— 3) MixedLM 不加 vc + week
    try:
        md = smf.mixedlm(
            f1, data=data,
            groups=data["celeb_season_id"].values
        )
        res = md.fit(reml=True, method="lbfgs")
        results["judge_best"] = ("MixedLM_no_vc_week", f1, res)
        return results
    except Exception as e:
        results["judge_try3_error"] = repr(e)

    # —— 4) MixedLM 不加 vc（去 week）
    try:
        md = smf.mixedlm(
            f2, data=data,
            groups=data["celeb_season_id"].values
        )
        res = md.fit(reml=True, method="lbfgs")
        results["judge_best"] = ("MixedLM_no_vc_no_week", f2, res)
        return results
    except Exception as e:
        results["judge_try4_error"] = repr(e)

    # —— 5) OLS + 舞者固定效应 + 聚类稳健标准误（兜底）
    f5 = "J_total ~ celebrity_age_during_season + C(industry2) + C(homeland2) + C(season) + C(week) + C(ballroom_partner)"
    ols = smf.ols(f5, data=data).fit(
        cov_type="cluster",
        cov_kwds={"groups": data["celeb_season_id"]}
    )
    results["judge_best"] = ("OLS_partnerFE_cluster", f5, ols)
    return results


In [19]:
judge_out = fit_judge_models(df_model2)
judge_kind, judge_formula_used, judge_res = judge_out["judge_best"]

print("=== Judge model chosen ===")
print("Type:", judge_kind)
print("Formula:", judge_formula_used)
print(judge_res.summary())

# 如果不是 MixedLM，看看你之前失败原因（可选）
for k, v in judge_out.items():
    if "error" in k:
        print(k, ":", v)


/Users/ronnygottheway/miniforge3/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)


=== Judge model chosen ===
Type: MixedLM_no_vc_week
Formula: J_total ~ celebrity_age_during_season + C(industry2) + C(homeland2) + C(season) + C(week)
                        Mixed Linear Model Regression Results
Model:                       MixedLM          Dependent Variable:          J_total   
No. Observations:            2770             Method:                      REML      
No. Groups:                  420              Scale:                       11.9536   
Min. group size:             1                Log-Likelihood:              -7441.8143
Max. group size:             11               Converged:                   Yes       
Mean group size:             6.6                                                     
-------------------------------------------------------------------------------------
                                         Coef.  Std.Err.    z    P>|z|  [0.025 0.975]
-------------------------------------------------------------------------------------
Intercept    

In [20]:
def fit_fan_models(data):
    results = {}

    # 1) MixedLM + vc + week
    f1 = "fan_logit ~ celebrity_age_during_season + C(industry2) + C(homeland2) + C(season) + C(week)"
    try:
        md = smf.mixedlm(
            f1, data=data,
            groups=data["celeb_season_id"].values,
            vc_formula={"partner_re": "0 + C(ballroom_partner)"}
        )
        res = md.fit(reml=True, method="lbfgs")
        results["fan_best"] = ("MixedLM_vc_week", f1, res)
        return results
    except Exception as e:
        results["fan_try1_error"] = repr(e)

    # 2) MixedLM + vc（去 week）
    f2 = "fan_logit ~ celebrity_age_during_season + C(industry2) + C(homeland2) + C(season)"
    try:
        md = smf.mixedlm(
            f2, data=data,
            groups=data["celeb_season_id"].values,
            vc_formula={"partner_re": "0 + C(ballroom_partner)"}
        )
        res = md.fit(reml=True, method="lbfgs")
        results["fan_best"] = ("MixedLM_vc_no_week", f2, res)
        return results
    except Exception as e:
        results["fan_try2_error"] = repr(e)

    # 3) MixedLM 不加 vc + week
    try:
        md = smf.mixedlm(
            f1, data=data,
            groups=data["celeb_season_id"].values
        )
        res = md.fit(reml=True, method="lbfgs")
        results["fan_best"] = ("MixedLM_no_vc_week", f1, res)
        return results
    except Exception as e:
        results["fan_try3_error"] = repr(e)

    # 4) MixedLM 不加 vc（去 week）
    try:
        md = smf.mixedlm(
            f2, data=data,
            groups=data["celeb_season_id"].values
        )
        res = md.fit(reml=True, method="lbfgs")
        results["fan_best"] = ("MixedLM_no_vc_no_week", f2, res)
        return results
    except Exception as e:
        results["fan_try4_error"] = repr(e)

    # 5) OLS + 舞者固定效应 + 聚类稳健标准误（兜底）
    f5 = "fan_logit ~ celebrity_age_during_season + C(industry2) + C(homeland2) + C(season) + C(week) + C(ballroom_partner)"
    ols = smf.ols(f5, data=data).fit(
        cov_type="cluster",
        cov_kwds={"groups": data["celeb_season_id"]}
    )
    results["fan_best"] = ("OLS_partnerFE_cluster", f5, ols)
    return results


fan_out = fit_fan_models(df_model2)
fan_kind, fan_formula_used, fan_res = fan_out["fan_best"]

print("=== Fan model chosen ===")
print("Type:", fan_kind)
print("Formula:", fan_formula_used)
print(fan_res.summary())

for k, v in fan_out.items():
    if "error" in k:
        print(k, ":", v)


=== Fan model chosen ===
Type: MixedLM_vc_week
Formula: fan_logit ~ celebrity_age_during_season + C(industry2) + C(homeland2) + C(season) + C(week)
                       Mixed Linear Model Regression Results
Model:                      MixedLM          Dependent Variable:          fan_logit
No. Observations:           2770             Method:                      REML     
No. Groups:                 420              Scale:                       0.0463   
Min. group size:            1                Log-Likelihood:              -561.6904
Max. group size:            11               Converged:                   Yes      
Mean group size:            6.6                                                    
-----------------------------------------------------------------------------------
                                         Coef.  Std.Err.   z    P>|z| [0.025 0.975]
-----------------------------------------------------------------------------------
Intercept                          

In [21]:
beta_j = judge_res.params
beta_f = fan_res.params

common = beta_j.index.intersection(beta_f.index)

comp = pd.DataFrame({
    "beta_judge": beta_j[common],
    "beta_fan": beta_f[common],
})
comp["delta(judge-fan)"] = comp["beta_judge"] - comp["beta_fan"]

# 只看你关心的（年龄/行业/地区）
focus = [idx for idx in comp.index if (
    ("celebrity_age_during_season" in idx) or
    ("C(industry2)" in idx) or
    ("C(homeland2)" in idx)
)]

comp_focus = comp.loc[focus].sort_values("delta(judge-fan)", ascending=False)
print(comp_focus.head(30))


                                          beta_judge  beta_fan  \
C(industry2)[T.Conservationist]             3.169895  0.049973   
C(homeland2)[T.Maine]                       4.100772  1.525605   
C(industry2)[T.Social media personality]    2.256517 -0.225931   
C(industry2)[T.Musician]                    2.582645  0.672511   
C(industry2)[T.Producer]                    2.490966  0.774863   
C(homeland2)[T.Nevada]                      1.243377  0.239297   
C(homeland2)[T.Russia]                      0.276078 -0.722559   
C(homeland2)[T.Nebraska]                    0.464856 -0.388586   
C(homeland2)[T.South Korea]                 1.440361  0.637414   
C(industry2)[T.News Anchor]                 0.955953  0.270593   
C(homeland2)[T.Hawaii]                      0.313114 -0.299861   
C(homeland2)[T.France]                      1.242111  0.724331   
C(industry2)[T.Social Media Personality]    0.484801 -0.005502   
C(homeland2)[T.Minnesota]                   0.246018 -0.230875   
C(homeland

In [22]:
hazard_formula = """
if_elim ~ J_total + fan_share_t
         + celebrity_age_during_season
         + C(industry2) + C(homeland2)
         + C(season) + C(week)
"""

haz = smf.glm(
    hazard_formula,
    data=df_model2,
    family=sm.families.Binomial()
).fit(cov_type="cluster", cov_kwds={"groups": df_model2["celeb_season_id"]})

print(haz.summary())


                 Generalized Linear Model Regression Results                  
Dep. Variable:                if_elim   No. Observations:                 2770
Model:                            GLM   Df Residuals:                     2653
Model Family:                Binomial   Df Model:                          116
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -613.60
Date:                Sun, 01 Feb 2026   Deviance:                       1227.2
Time:                        18:04:45   Pearson chi2:                 2.12e+03
No. Iterations:                    24   Pseudo R-squ. (CS):             0.2180
Covariance Type:              cluster                                         
                                               coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------

In [23]:
import numpy as np

bJ = haz.params["J_total"]
bF = haz.params["fan_share_t"]

for dp in [0.01, 0.05, 0.10]:
    dlogit = bF * dp
    odds_ratio = np.exp(dlogit)
    print(f"fan_share +{dp:.2f}: Δlogit={dlogit:.3f}, odds×={odds_ratio:.3f}")

# J_total 每+1分的 odds ratio
print("J_total +1: odds×=", np.exp(bJ))


fan_share +0.01: Δlogit=-0.286, odds×=0.751
fan_share +0.05: Δlogit=-1.431, odds×=0.239
fan_share +0.10: Δlogit=-2.861, odds×=0.057
J_total +1: odds×= 0.8426020266891393


In [25]:
fan_fe_partner = smf.ols(
    "fan_logit ~ celebrity_age_during_season + C(industry2) + C(homeland2) + C(season) + C(week) + C(ballroom_partner)",
    data=df_model2
).fit(cov_type="cluster", cov_kwds={"groups": df_model2["celeb_season_id"]})

# 舞者系数提取（相对 baseline 舞者）
partner_coef = fan_fe_partner.params.filter(like="C(ballroom_partner)")
partner_tab = partner_coef.sort_values(ascending=False).reset_index()
partner_tab.columns = ["partner_term", "coef"]
print(partner_tab.head(20))


                                         partner_term      coef
0               C(ballroom_partner)[T.Tyne Stecklein]  0.974192
1   C(ballroom_partner)[T.Daniella Karagach (Rumer...  0.421152
2                 C(ballroom_partner)[T.Koko Iwasaki]  0.329133
3                  C(ballroom_partner)[T.Derek Hough]  0.206062
4   C(ballroom_partner)[T.Emma Slater/Kaitlyn Bris...  0.089771
5   C(ballroom_partner)[T.Witney Carson (Xoshitl G...  0.039286
6                 C(ballroom_partner)[T.Corky Ballas]  0.013281
7                C(ballroom_partner)[T.Pasha Pashkov] -0.056667
8                  C(ballroom_partner)[T.Emma Slater] -0.059054
9                 C(ballroom_partner)[T.Cheryl Burke] -0.172107
10       C(ballroom_partner)[T.Valentin Chmerkovskiy] -0.273152
11               C(ballroom_partner)[T.Witney Carson] -0.278216
12  C(ballroom_partner)[T.Val Chmerkovskiy (Joey G... -0.328725
13                 C(ballroom_partner)[T.Mark Ballas] -0.336656
14              C(ballroom_partner)[T.He

In [26]:
# comp_focus 已经是年龄+行业+地区的对比表
comp_focus = comp_focus.copy()
comp_focus["abs_delta"] = comp_focus["delta(judge-fan)"].abs()

top15 = comp_focus.sort_values("abs_delta", ascending=False).head(15)
print(top15)


                                          beta_judge  beta_fan  \
C(industry2)[T.Sports Broadcaster]         -4.965289 -0.663639   
C(homeland2)[T.Italy]                      -4.287774 -0.212180   
C(homeland2)[T.Alaska]                     -4.135698 -0.450370   
C(industry2)[T.Conservationist]             3.169895  0.049973   
C(industry2)[T.Other]                      -2.983961 -0.342173   
C(industry2)[T.Entrepreneur]               -1.975512  0.665311   
C(industry2)[T.Motivational Speaker]       -2.374902  0.256872   
C(homeland2)[T.Maine]                       4.100772  1.525605   
C(industry2)[T.Radio Personality]          -2.301278  0.221695   
C(industry2)[T.Social media personality]    2.256517 -0.225931   
C(homeland2)[T.Australia]                  -2.622739 -0.153029   
C(industry2)[T.Military]                   -1.966366  0.463851   
C(homeland2)[T.Kentucky]                   -2.863815 -0.441832   
C(homeland2)[T.Louisiana]                  -2.203186  0.187538   
C(homeland

In [27]:
# comp_focus 里已经有 delta 和 abs_delta
top_judge = comp_focus.sort_values("delta(judge-fan)", ascending=False).head(10)
top_fan   = comp_focus.sort_values("delta(judge-fan)", ascending=True).head(10)

print("=== More Judge-favored (Δ large positive) ===")
print(top_judge[["beta_judge","beta_fan","delta(judge-fan)","abs_delta"]])

print("\n=== More Fan-favored (Δ large negative) ===")
print(top_fan[["beta_judge","beta_fan","delta(judge-fan)","abs_delta"]])


=== More Judge-favored (Δ large positive) ===
                                          beta_judge  beta_fan  \
C(industry2)[T.Conservationist]             3.169895  0.049973   
C(homeland2)[T.Maine]                       4.100772  1.525605   
C(industry2)[T.Social media personality]    2.256517 -0.225931   
C(industry2)[T.Musician]                    2.582645  0.672511   
C(industry2)[T.Producer]                    2.490966  0.774863   
C(homeland2)[T.Nevada]                      1.243377  0.239297   
C(homeland2)[T.Russia]                      0.276078 -0.722559   
C(homeland2)[T.Nebraska]                    0.464856 -0.388586   
C(homeland2)[T.South Korea]                 1.440361  0.637414   
C(industry2)[T.News Anchor]                 0.955953  0.270593   

                                          delta(judge-fan)  abs_delta  
C(industry2)[T.Conservationist]                   3.119922   3.119922  
C(homeland2)[T.Maine]                             2.575167   2.575167  
C(industry2